# CSD 2026 — customer effort, triangulated with validation rules and rejection reasons

[Notebook 08](08_csd_2026_validation_rules.ipynb) tagged 12,000 Scopus Content Service Desk tickets
with the validation rule that made each request invalid and the rejection reason the agent gave.
This notebook adds the third axis: **how much effort the interaction cost the customer** — and
joins it to the other two.

## Two independent readings of the same interaction

| | source | what it gives |
|---|---|---|
| **Observed effort** | the thread record | turns, agent handoffs, wait time, repeat contact. Exact, free, no model involved. |
| **Judged effort** | the transcript, read by the model | effort score, friction points, effort themes, resolution transparency. Judgement the record cannot supply. |

Neither depends on anything outside this corpus. They are computed separately and compared in
section 9 — where they agree, that is corroboration; where they diverge, that is a finding.

## Reusing what exists

The `customer_effort` module (prefix `CE`, 20-node taxonomy, nine hand-tuned field prompts) already
exists in `definitions/scopus_content_pipeline_definition_v2.xlsx`. It is reused essentially as
written. Three changes, each recorded where it is made:

1. **The module is renamed `csd_customer_effort`** and given its own taxonomy file, so it cannot
   collide with the Scopus-tuned original. The `CE` prefix is deliberately **kept** — the
   `effort_themes` prompt cites `CE2.1` / `CE3.2` by name, and renaming the prefix would silently
   invalidate its own examples.
2. **`channel_switches` is dropped.** The channel never varies within a ticket in this corpus
   (0 of 56,935), and true channel redirects appear in about 1% of transcripts.
3. **A ninth friction category is added — "Redirected to another party".** Being sent to the
   publisher, the journal or another team appears in ~11.7% of these tickets and none of the
   original eight categories covers it. On a corpus that is entirely rejections, being redirected
   *is* the outcome for a large minority of customers.

## What this is not

There is no customer-reported effort score for these tickets. Survey data is expected and section
14 is built to receive it, but until it lands **nothing here is validated against a customer**.
The observed/judged comparison is corroboration between two views, not ground truth.

## Cost

| Section | | Cost |
|---|---|---|
| 1–5 | workbook, build, observed features | free |
| 7 | 40-ticket smoke | ~$0.60 |
| 8 | 1,000-ticket probe | ~$7 |
| 10 | main run, 12,000 tickets | ~$82 |
| 11–14 | triangulation, analysis, deliverable | free |


## 0. Configuration

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
DATA_DIR   = BASE / 'data' / 'scopus content'
META_TSV   = DATA_DIR / 'scopus content_2026_metadata.tsv'
SOURCE_DEFN = BASE / 'definitions' / 'scopus_content_pipeline_definition_v2.xlsx'
SOURCE_MODULE = 'customer_effort'          # the module being adapted

# Written by notebook 08 - the same 12,000 rows, so every join is exact
SAMPLE_CSV = BASE / 'runs' / 'intermediate' / 'csd_sample_12000.csv'
STAGE_INPUT_08 = BASE / 'data' / 'csd_stage_input_2026.csv'
TAGGED_XLSX = BASE / 'runs' / 'output' / 'csd_2026_validation_rules.xlsx'

# Written by scripts/build_effort_features.py
FEATURES_CSV = BASE / 'runs' / 'intermediate' / 'csd_effort_features.csv'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX    = BASE / 'definitions' / 'csd_2026_pipeline_definition.xlsx'
TAXONOMY_DIR = BASE / 'generated' / 'taxonomies'
CONFIG_JSON  = BASE / 'generated' / 'pipeline_config.json'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_csd_effort'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'
RC_LABEL = 'rc_label'          # ground truth, from notebook 08's output

# -- The module ------------------------------------------------------------
MOD_EFFORT   = 'csd_customer_effort'
PREFIX_EFFORT = 'CE'           # kept, NOT renamed - see the header note
DROP_FIELDS  = ['channel_switches']

# The three modules notebook 08 built. They must survive this rebuild.
MOD_RULE, MOD_REASON, MOD_PROPOSAL = (
    'csd_validation_rule', 'csd_rejection_reason', 'csd_rule_proposal')
EXISTING_MODULES = [MOD_RULE, MOD_REASON, MOD_PROPOSAL]
ALL_MODULES = EXISTING_MODULES + [MOD_EFFORT]

# -- Sampling --------------------------------------------------------------
SMOKE_N, PROBE_N, RANDOM_SEED = 40, 1_000, 0

# -- Outputs ---------------------------------------------------------------
OUT_DIR   = BASE / 'runs' / 'output'
INTER_DIR = BASE / 'runs' / 'intermediate'
SMOKE_DIR = BASE / 'runs' / 'smoke_csd_2026'
PROFILE_DIR = BASE / 'profiles'
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'
for _d in (OUT_DIR, INTER_DIR, SMOKE_DIR, PROFILE_DIR, CHECKPOINT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# csd_* stems so the checkpoint cannot collide with another notebook's
EFFORT_SMOKE_CSV = SMOKE_DIR / 'csd_effort_smoke_input.csv'
EFFORT_PROBE_CSV = SMOKE_DIR / 'csd_effort_probe_input.csv'
PROFILE_EFFORT   = PROFILE_DIR / 'csd_2026_customer_effort.json'
EFFORT_OUT       = OUT_DIR / 'csd_2026_customer_effort.xlsx'
TRIANGULATION_XLSX = OUT_DIR / 'csd_2026_effort_triangulation.xlsx'
SURVEY_FILE = None      # set in section 13 when the survey data arrives

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONUTF8'] = '1'
SUBPROCESS_ENV['PYTHONIOENCODING'] = 'utf-8'

for _p in (SOURCE_DEFN, DEFN_XLSX, SAMPLE_CSV, TAGGED_XLSX, FEATURES_CSV):
    assert _p.exists(), (
        f'Missing: {_p}\n'
        f'  csd_effort_features.csv comes from:  '
        f'.venv/Scripts/python.exe scripts/build_effort_features.py\n'
        f'  the others come from notebook 08.')
print('Project root :', BASE)
print('pandas       :', pd.__version__)
print('Modules after this build:', ', '.join(ALL_MODULES))
print('Paths OK')

## 0b. Shared helpers

Copied from [notebook 08](08_csd_2026_validation_rules.ipynb) — they are notebook-local, not
library code. `run_stage()` and the checkpoint fingerprint guard are the load-bearing ones: the
pipeline resumes silently whenever every module already has a result for a record, which is right
after an interruption and wrong after a prompt edit.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '​‌‍﻿'))


def norm_label(value) -> str:
    """Normalise a label to a stable join key, preserving wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'\s+', ' ', text).strip()


def norm_prose(value) -> str:
    return norm_label(value)


def norm_cmp(value) -> str:
    return norm_label(value).casefold()


def read_stage_output(path) -> pd.DataFrame:
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


def stratified_sample(df: pd.DataFrame, by: str, n: int, *, min_per=0, max_per=None,
                      seed=RANDOM_SEED) -> pd.DataFrame:
    """Sample ~n rows proportional by `by`, with a floor per stratum."""
    groups = {k: g for k, g in df.groupby(by, dropna=False)}
    take = {k: min(len(g), min_per) for k, g in groups.items()}
    remaining = max(0, n - sum(take.values()))
    slack = {k: len(g) - take[k] for k, g in groups.items()}
    total_slack = sum(slack.values())
    if remaining and total_slack:
        for k, g in groups.items():
            take[k] = min(len(g), take[k] + int(round(remaining * slack[k] / total_slack)))
    if max_per is not None:
        take = {k: min(v, max_per) for k, v in take.items()}
    out = pd.concat([g.sample(n=take[k], random_state=seed)
                     for k, g in groups.items() if take[k]], ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


print('Helpers defined.')

## 0c. Back up the shared config

`rca build build` regenerates `generated/pipeline_config.json` from whichever workbook it
is handed, and does not merge. That is why the effort module is being added **into** the CSD
workbook rather than built from the Scopus one — a build from
`scopus_content_pipeline_definition_v2.xlsx` would evict the three modules notebook 08 depends on.

This cell snapshots the current config so the pre-effort state is recoverable, and records what it
found.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_before = json.loads(CONFIG_JSON.read_text(encoding='utf-8'))
_backed = []
for _src in [CONFIG_JSON, BASE / 'generated' / 'build_manifest.json',
             DEFN_XLSX, *sorted(TAXONOMY_DIR.glob('csd_*.json'))]:
    _dst = CONFIG_BACKUP / _src.name
    if _src.exists() and not _dst.exists():
        shutil.copy2(_src, _dst)
        _backed.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}  ({len(_backed)} file(s) copied)')
print(f'\nModules in the live config before this notebook: {sorted(_before["modules"])}')
# Subset, not equality: on a re-run the effort module is already there, and that
# is fine. What must hold is that notebook 08's three modules are still present -
# if they are not, some other build has evicted them and rebuilding here would
# bake in their absence.
_missing = set(EXISTING_MODULES) - set(_before['modules'])
assert not _missing, (
    f'notebook 08 modules missing from the live config: {sorted(_missing)}. Something else '
    f'has rebuilt over them. Re-run notebook 08 section 9 before continuing.')
if MOD_EFFORT in _before['modules']:
    print(f'  ({MOD_EFFORT} is already registered - this is a re-run)')

(CONFIG_BACKUP / 'RESTORE.txt').write_text(
    'Snapshot of generated/pipeline_config.json and definitions/csd_2026_pipeline_definition.xlsx\n'
    'taken before notebook 09 added the csd_customer_effort module.\n\n'
    'To go back:\n'
    f'    copy the .xlsx here over definitions/{DEFN_XLSX.name}\n'
    '    rca build --input definitions/'
    f'{DEFN_XLSX.name} --force-all\n', encoding='utf-8')
print(f'\nRestore instructions: {(CONFIG_BACKUP / "RESTORE.txt").relative_to(BASE)}')

## 1. Load the observed effort features

Produced by `scripts/build_effort_features.py` from
`data/scopus content/scopus content_2026_threads.tsv` — 176,083 thread entries across all 56,935
incidents. No model involved; these are counts and timestamps.

Two honest limits are carried in the data rather than papered over:

- `obs_repeat_contact` is `<NA>`, never `False`, for the ~13% of incidents whose customer cannot be
  identified. `CONTACT_ID` is only recorded on customer-authored entries, so "we do not know" and
  "no" are different facts and are stored differently.
- `obs_effort_index` is an **authored** equal-weight composite of four percentile-ranked
  components, not a fitted model. Its components are all kept so it can be reweighted rather than
  rebuilt.

**A warning about that index, established in section 13 and worth carrying forward.** Against the
customer's own effort answer it scores a Spearman of about **+0.04** — effectively nothing. Two of
its four components are *negatively* signed against the survey: agent handoffs and repeat contact,
the two metrics an operations team would most readily reach for. No operational signal measured
here exceeds **+0.09**.

So the index describes how heavy the *handling* was, and that turns out to be close to unrelated to
how hard the *customer* found it. It is kept because it is exact, free and useful for describing
workload — but it is not a proxy for customer effort, and nothing downstream treats it as one.

In [ ]:
features = pd.read_csv(FEATURES_CSV)
features[ID_COL] = features[ID_COL].astype(str).str.strip()
print(f'{FEATURES_CSV.name}: {len(features):,} incidents x {len(features.columns)} cols')
print('columns:', [c for c in features.columns if c.startswith('obs_')])

OBS_COLS = [c for c in features.columns if c.startswith('obs_')]
assert all(c.startswith('obs_') for c in features.columns if c != ID_COL), (
    'every feature column must be obs_-prefixed so it cannot collide with an LLM field '
    'or be mistaken for stale enrichment by drop_enrichment_cols()')

sample = pd.read_csv(SAMPLE_CSV, dtype=str)
sample[ID_COL] = sample[ID_COL].astype(str).str.strip()
_cov = sample[ID_COL].isin(set(features[ID_COL])).mean()
print(f'\nfeature coverage of the 12,000 sample: {_cov:.1%}')
assert _cov == 1.0, 'the observed features do not cover every sampled ticket'

print('\nDistribution across all 56,935 incidents:')
print(features[['obs_turns', 'obs_customer_turns', 'obs_agent_handoffs',
                'obs_first_response_h', 'obs_span_h', 'obs_effort_index']]
      .describe(percentiles=[.25, .5, .75, .9]).T
      .to_string(float_format=lambda v: f'{v:,.2f}'))
_rc = features['obs_repeat_contact']
print(f'\nobs_repeat_contact: True {int((_rc == True).sum()):,}  '
      f'False {int((_rc == False).sum()):,}  unknown {int(_rc.isna().sum()):,}')
print(f'\nOnly {(features.obs_customer_turns > 1).mean():.1%} of incidents have more than one '
      f'customer turn.')
print('That caps how much the observed signals can spread, and it is why the index is reported '
      'with its\ncomponents rather than as a single headline number.')

## 2. Read the effort module out of the Scopus workbook

The module is lifted wholesale — nine field prompts and a 20-node taxonomy, already tuned on
Scopus Content tickets. This cell reads it and shows exactly what was found, before anything is
changed.

In [ ]:
src_modules = pd.read_excel(SOURCE_DEFN, sheet_name='modules', dtype=str)
src_fields = pd.read_excel(SOURCE_DEFN, sheet_name='fields', dtype=str)
src_tax = pd.read_excel(SOURCE_DEFN, sheet_name='taxonomy', dtype=str)

ce_module = src_modules[src_modules.module_name == SOURCE_MODULE]
ce_fields = src_fields[src_fields.module_name == SOURCE_MODULE].copy()
ce_tax = src_tax[src_tax.module_name == SOURCE_MODULE].copy()
assert len(ce_module) == 1, f'expected exactly one {SOURCE_MODULE} module row'

print(f'Source: {SOURCE_DEFN.name}')
print(f'  module   : {SOURCE_MODULE}  (prefix {ce_module.iloc[0].taxonomy_prefix})')
print(f'  fields   : {len(ce_fields)}  -> {list(ce_fields.field_name)}')
print(f'  taxonomy : {int((ce_tax.level == "L1").sum())} L1 / '
      f'{int((ce_tax.level == "L2").sum())} L2')
print('\nTaxonomy:')
for _, r in ce_tax.iterrows():
    print(f'  {"  " if r.level == "L2" else ""}{r.level}  {r.label}')

## 3. Adapt it

Three changes, each asserted so it cannot silently fail to apply.

**Why the prefix stays `CE`.** The `effort_themes` prompt instructs the model to *"Return the codes
as a list: ['CE2.1', 'CE3.2']"*. Renaming the prefix would leave a tuned prompt citing codes that
no longer exist — the kind of change that looks tidy and quietly degrades accuracy. The module and
taxonomy file are renamed instead, which is enough to keep it separate from the Scopus original.

**The ninth friction category.** The original eight cover repetition, self-service failure,
escalation, waiting, transparency, wrong information, complexity and non-resolution. None covers
being told to take the request somewhere else — which happens in ~11.7% of these tickets, mostly
"contact the publisher". On a corpus that is entirely rejections that is not a detail.

In [ ]:
# ── the ninth category, and the keyword patterns that normalise variants ──
# normalise_friction_value() passes exact label matches straight through; these
# regexes only catch wording the model invents instead of using the label.
FRICTION_CATEGORIES = [
    ('Repeated issue explanation',    [r'repeat', r're-?explain', r'explain(ed)? again',
                                       r'transferr?ed']),
    ('Self-service failure',          [r'self-?serv', r'(form|tool|website|portal).{0,20}'
                                       r'(fail|not work|did ?n.?t work|error)']),
    ('Escalation required',           [r'escalat']),
    ('Slow response / long wait',     [r'slow', r'\bwait', r'delay', r'long time', r'chas(e|ing)',
                                       r'follow.?up']),
    ('Lack of transparency',          [r'transparen', r'unclear', r'no (status|update|visibility)',
                                       r'not (clear|informed)']),
    ('Incorrect or missing information', [r'incorrect', r'wrong', r'inaccurate',
                                          r'missing (info|data)', r'misinform']),
    ('Process complexity',            [r'complex', r'complicat', r'many steps', r'multi-?step',
                                       r'cumbersome']),
    ('Unresolved issue',              [r'unresolved', r'not resolved', r'no resolution',
                                       r'without (a )?resolution']),
    # -- the addition --
    ('Redirected to another party',   [r'redirect', r'referred', r'contact the (publisher|journal|'
                                       r'editor)', r'third.?party', r'another (team|party|'
                                       r'department)', r'librarian', r'sent (to|elsewhere)']),
]
NEW_CATEGORY = FRICTION_CATEGORIES[-1][0]
CANONICAL_FRICTION = [c for c, _ in FRICTION_CATEGORIES]

# ── adapt the module row ──────────────────────────────────────────────────
mod_row = ce_module.iloc[0].to_dict()
mod_row['module_name'] = MOD_EFFORT
mod_row['taxonomy_file'] = f'{MOD_EFFORT}.json'
mod_row['taxonomy_prefix'] = PREFIX_EFFORT
mod_row['reorder_group'] = 'customer_effort'
mod_row['reorder_order'] = 4
mod_row['enabled'] = True
mod_row['discovery_mode'] = ''        # the taxonomy is reused, not discovered
mod_row['extract_prompt'] = ''
mod_row['notes'] = (f'Adapted from {SOURCE_MODULE} in {SOURCE_DEFN.name} by notebook 09. '
                    f'channel_switches dropped; "{NEW_CATEGORY}" added to friction_points.')

# ── adapt the fields ──────────────────────────────────────────────────────
eff_fields = ce_fields[~ce_fields.field_name.isin(DROP_FIELDS)].copy()
assert len(eff_fields) == len(ce_fields) - len(DROP_FIELDS), 'a dropped field was not found'
eff_fields['module_name'] = MOD_EFFORT

# Extend the friction_points instruction with the ninth category, inserted before
# the closing sentence so the wording still reads as one list.
_fp = eff_fields.field_name == 'friction_points'
assert _fp.sum() == 1
_old = eff_fields.loc[_fp, 'prompt_instruction'].iloc[0]
_anchor = "'Unresolved issue' (interaction ended without resolution)."
assert _anchor in _old, 'the friction_points instruction has changed shape; re-read it'
_new = _old.replace(
    _anchor,
    _anchor[:-1] + f", '{NEW_CATEGORY}' (customer was told to take the request to the publisher, "
    f"the journal, another team, or any party outside this service desk).")
# The 40-ticket smoke returned 'Follow-up required by customer' in friction_points -
# that is a taxonomy LEAF LABEL, not a friction category. The model was drawing from
# the wrong list. Close it off explicitly.
_new += (" Use ONLY the nine labels above, copied exactly. Do NOT put taxonomy codes or "
         "taxonomy leaf labels in this field - effort_themes is where those belong.")
eff_fields.loc[_fp, 'prompt_instruction'] = _new

# ── enum options must be lowercase ────────────────────────────────────────
# The generated validator lowercases the model's answer but compares it against
# the raw option text, so a capitalised option rejects every value and blanks the
# column. Check rather than trust.
for _, r in eff_fields.iterrows():
    ft = str(r.field_type)
    if ft.startswith('enum:'):
        opts = [o.strip() for o in ft[5:].split(',')]
        bad = [o for o in opts if o != o.lower()]
        assert not bad, (f'{r.field_name}: enum options must be lowercase or the generated '
                         f'validator blanks every value. Offending: {bad}')
        print(f'  enum ok: {r.field_name} -> {opts}')

eff_tax = ce_tax.copy()
eff_tax['module_name'] = MOD_EFFORT

print(f'\nAdapted module: {MOD_EFFORT} (prefix {PREFIX_EFFORT}, '
      f'taxonomy {mod_row["taxonomy_file"]})')
print(f'  fields   : {len(eff_fields)} (dropped {DROP_FIELDS})')
print(f'  taxonomy : {len(eff_tax)} rows, unchanged')
print(f'  friction : {len(CANONICAL_FRICTION)} categories, added "{NEW_CATEGORY}"')
print(f'\nNew friction_points instruction:\n{_new}')

## 4. Write the workbook and build

The effort module is **appended** to the CSD workbook, leaving the three modules notebook 08 built
untouched. After the build the config holds four modules and both notebooks remain runnable.

Two checks worth their space:

- **No field-name collisions across modules.** Results from every module are merged per uid with
  `dict.update()` (`rca run:890-894`), so two modules declaring the same field name silently
  overwrite each other with whichever finished last. Nothing in the build warns about this.
- **`friction_categories` is written into the config.** `build_pipeline_config()` never emits this
  key, so `normalise_friction_columns()` is a no-op in every config in this repo — friction points
  come back in whatever words the model chose. The categories are written in after the build, and
  re-written after any later rebuild, because a rebuild wipes them.

In [ ]:
modules_df = pd.read_excel(DEFN_XLSX, sheet_name='modules', dtype=str)
fields_df = pd.read_excel(DEFN_XLSX, sheet_name='fields', dtype=str)
taxonomy_df = pd.read_excel(DEFN_XLSX, sheet_name='taxonomy', dtype=str)
print(f'{DEFN_XLSX.name} before: {len(modules_df)} modules, {len(fields_df)} fields, '
      f'{len(taxonomy_df)} taxonomy rows')

# Idempotent: drop any previous copy of the effort module before appending.
modules_df = modules_df[modules_df.module_name != MOD_EFFORT]
fields_df = fields_df[fields_df.module_name != MOD_EFFORT]
taxonomy_df = taxonomy_df[taxonomy_df.module_name != MOD_EFFORT]

modules_out = pd.concat([modules_df, pd.DataFrame([mod_row])], ignore_index=True)
fields_out = pd.concat([fields_df, eff_fields], ignore_index=True)
taxonomy_out = pd.concat([taxonomy_df, eff_tax], ignore_index=True)

# ---- field-name collisions across ALL modules ---------------------------
_dupes = [f for f, n in Counter(fields_out.field_name).items() if n > 1]
assert not _dupes, (
    f'these field names appear in more than one module and would silently clobber each other '
    f'at merge time: {_dupes}')

# ---- structural-prefix collisions ---------------------------------------
# drop_enrichment_cols() strips any column starting with a structural prefix from
# a pipeline INPUT, to purge stale enrichment before a re-run.
#
# `resolution_transparency` starts with `resolution_` and is therefore caught. It
# is allowed through deliberately: the name is part of the established effort
# schema that downstream tooling already reads, and renaming it here would be a
# silent breaking change. It is safe in this pipeline because the stage input is
# only [INCIDENT_REF, Conversation] - asserted in section 6 - so there is nothing
# for drop_enrichment_cols() to strip.
#
# What WOULD break: feeding csd_2026_customer_effort.xlsx back in as an input.
# The column would vanish without a word. Do not do that.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
KNOWN_PREFIX_EXCEPTIONS = {'resolution_transparency'}
for _f in fields_out.field_name:
    if str(_f) in KNOWN_PREFIX_EXCEPTIONS:
        continue
    _bad = [p for p in _STRUCTURAL_PREFIXES if str(_f).startswith(p)]
    assert not _bad, f'field {_f!r} starts with structural prefix {_bad} and would be dropped'
_flagged = sorted(f for f in fields_out.field_name if str(f) in KNOWN_PREFIX_EXCEPTIONS)
if _flagged:
    print(f'  NOTE: {_flagged} collide(s) with a structural prefix and are allowed through '
          f'by exception -\n        safe only because the stage input carries no enrichment '
          f'columns. See the comment above.')

with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as w:
    modules_out.to_excel(w, sheet_name='modules', index=False)
    fields_out.to_excel(w, sheet_name='fields', index=False)
    taxonomy_out.to_excel(w, sheet_name='taxonomy', index=False)

print(f'{DEFN_XLSX.name} after : {len(modules_out)} modules, {len(fields_out)} fields, '
      f'{len(taxonomy_out)} taxonomy rows')
print(f'  modules: {list(modules_out.module_name)}')
print(f'  no field-name collisions across {len(fields_out)} fields')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force-all'], 'build --force-all')

cfg = json.loads(CONFIG_JSON.read_text(encoding='utf-8'))
print('\nModules now in the live config:', sorted(cfg['modules']))
assert set(cfg['modules']) == set(ALL_MODULES), (
    f'expected {sorted(ALL_MODULES)}, got {sorted(cfg["modules"])}')

_tax = json.loads((TAXONOMY_DIR / f'{MOD_EFFORT}.json').read_text(encoding='utf-8'))
_leaves = {c: leaf['label'] for node in _tax.values()
           for c, leaf in node.get('children', {}).items()}
print(f'{MOD_EFFORT}.json: {len(_tax)} L1 groups, {len(_leaves)} leaves')
EFFORT_LEAVES = _leaves

# The effort_themes prompt names CE2.1 and CE3.2 explicitly - they must exist.
for _cited in ('CE2.1', 'CE3.2'):
    assert _cited in _leaves, (
        f'the effort_themes prompt cites {_cited} but it is not in the taxonomy')
print(f'  codes cited by the effort_themes prompt (CE2.1, CE3.2) both exist')

In [ ]:
# ---- friction_categories: write them in, because the build never does ----
# _load_friction_categories() (rca_pipeline/utils.py) reads this key fresh on every
# call, so writing it here takes effect immediately - but a later rebuild wipes it,
# which is why this cell must be re-run after any rebuild.
cfg = json.loads(CONFIG_JSON.read_text(encoding='utf-8'))
cfg['friction_categories'] = [{'label': lab, 'keywords': pats}
                              for lab, pats in FRICTION_CATEGORIES]
CONFIG_JSON.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding='utf-8')

import importlib
import rca_pipeline.utils as _u
importlib.reload(_u)
_cats = _u._get_friction_categories()
print(f'friction_categories now in the config: {len(_cats)}')
assert len(_cats) == len(FRICTION_CATEGORIES), 'the config did not take the categories'

# Prove normalisation actually fires, including on the new category.
_probes = [
    ('Repeated issue explanation', 'Repeated issue explanation'),   # exact label passes through
    ('customer had to explain again after being transferred', 'Repeated issue explanation'),
    ('agent escalated to the content team', 'Escalation required'),
    ('customer was told to contact the publisher', NEW_CATEGORY),
    ('referred to a third-party site', NEW_CATEGORY),
    ('long wait for a reply', 'Slow response / long wait'),
]
print('\nnormalise_friction_value():')
for raw, want in _probes:
    got = _u.normalise_friction_value(raw)
    flag = 'ok ' if got == want else 'FAIL'
    print(f'  {flag} {raw[:52]:<54} -> {got}')
    assert got == want, f'{raw!r} normalised to {got!r}, expected {want!r}'
print('\nFriction normalisation is live. It is a no-op in every other config in this repo.')

## 5. The LLM input, the profile and the checkpoint guard

The effort run uses **its own copy** of the 12,000-row input, under a different filename. The rows
are identical to notebook 08's — copied, not re-sampled — so every join is exact, but the
checkpoint is separate. Sharing notebook 08's input file would mean sharing its checkpoint, and the
fingerprint guard would delete it the moment it saw a module it did not recognise.

The model sees the ticket id and the conversation. Nothing else: not the resolution code, not the
validation rule, not the rejection reason. Effort has to be read out of the interaction, or the
triangulation in section 11 is circular.

In [ ]:
LLM_COLS = [ID_COL, TEXT_COL]
EFFORT_INPUT = INTER_DIR / 'csd_effort_input_12000.csv'

_src = pd.read_csv(SAMPLE_CSV, dtype=str)
assert list(_src.columns) == LLM_COLS, f'unexpected columns in {SAMPLE_CSV.name}: {list(_src.columns)}'
_src.to_csv(EFFORT_INPUT, index=False)

# Identical rows to notebook 08, separate checkpoint.
tagged = read_stage_output(TAGGED_XLSX)
assert set(_src[ID_COL].astype(str).str.strip()) == set(tagged[ID_COL]), (
    'the effort input and the notebook 08 output cover different tickets - the join would be partial')

with open(EFFORT_INPUT, encoding='utf-8') as _f:
    _header = _f.readline().strip()
assert _header == ','.join(LLM_COLS), f'unexpected header on disk: {_header!r}'
_leak = [c for c in _src.columns if 'resolution' in c.lower() or 'rule' in c.lower()
         or 'reason' in c.lower()]
assert not _leak, f'ground-truth columns in the effort input: {_leak}'

print(f'Wrote {EFFORT_INPUT.relative_to(BASE)}')
print(f'  {len(_src):,} rows x {len(_src.columns)} cols: {LLM_COLS}')
print(f'  header on disk: {_header}')
print(f'  same {len(tagged):,} tickets as notebook 08 - joins will be exact')
print(f'\nEstimated cost at $0.0068/record: ${len(_src) * 0.0068:,.0f}')

In [ ]:
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}

TUNING = {'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4, 'retries': 5,
          'max_tokens': 8192, 'track_costs': True, 'cost_report_dir': 'runs/cost_reports'}

spec = {'_comment': ('Customer effort for the same 12,000 CSD 2026 tickets notebook 08 tagged '
                     'with validation rules and rejection reasons. Own input copy and own '
                     'checkpoint so it cannot disturb that run. Adapted from the customer_effort '
                     'module in scopus_content_pipeline_definition_v2.xlsx: channel_switches '
                     'dropped (channel never varies within a ticket in this corpus), ninth '
                     'friction category added.'),
        'input': EFFORT_INPUT.relative_to(BASE).as_posix(),
        'output': EFFORT_OUT.relative_to(BASE).as_posix(),
        'taxonomy_dir': 'generated/taxonomies',
        'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': [MOD_EFFORT], **TUNING}
_unknown = [k for k in spec if k != '_comment' and k not in _dests]
assert not _unknown, f'keys rca run would silently ignore: {_unknown}'
PROFILE_EFFORT.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {PROFILE_EFFORT.relative_to(BASE)}')

# The sentinel decides what "already done" means on resume. For this module it
# should be effort_score - the first field in the sheet and the first non-code
# field. Verify rather than assume: a wrong sentinel silently skips every record.
from rca_pipeline.runner import _build_sentinel_map
SENTINELS = _build_sentinel_map(json.loads(CONFIG_JSON.read_text(encoding='utf-8')))
print(f'\nsentinel for {MOD_EFFORT}: {SENTINELS.get(MOD_EFFORT)!r}')
assert SENTINELS.get(MOD_EFFORT) in ('effort_score', 'effort_primary_code'), (
    f'unexpected sentinel {SENTINELS.get(MOD_EFFORT)!r} - resume would misbehave')

In [ ]:
# ---- prompt fingerprint + checkpoint guard (adapted from notebook 08) ----
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
_name = f'generated.modules.{MOD_EFFORT}'
_mod = importlib.reload(sys.modules[_name]) if _name in sys.modules \
    else importlib.import_module(_name)
_tax = json.loads((TAXONOMY_DIR / f'{MOD_EFFORT}.json').read_text(encoding='utf-8'))
EFFORT_PROMPT = _mod._build_system_prompt(taxonomy_to_rich_prompt(_tax, PREFIX_EFFORT))

# Every taxonomy leaf and every friction category must survive the round trip
# through Excel, the taxonomy JSON and the prompt builder.
for _c, _lab in EFFORT_LEAVES.items():
    assert _lab in EFFORT_PROMPT, f'taxonomy leaf missing from the prompt: {_lab!r}'
for _cat in CANONICAL_FRICTION:
    assert _cat in EFFORT_PROMPT, f'friction category missing from the prompt: {_cat!r}'
assert 'channel' not in EFFORT_PROMPT.lower() or 'channel_switches' not in EFFORT_PROMPT, \
    'channel_switches survived into the prompt'
print(f'Prompt: {len(EFFORT_PROMPT):,} chars (~{len(EFFORT_PROMPT) // 4:,} tokens)')
print(f'  all {len(EFFORT_LEAVES)} taxonomy leaves present')
print(f'  all {len(CANONICAL_FRICTION)} friction categories present')
print(f'  channel_switches absent')

PROMPT_FINGERPRINTS = {
    MOD_EFFORT: hashlib.sha256(EFFORT_PROMPT.encode('utf-8')).hexdigest()[:16]}


def checkpoint_files(input_path) -> list:
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, modules) -> None:
    """Discard the checkpoint if any of these modules' prompts have changed.

    Resume is the default: the pipeline skips any record that already has a value
    for the module's sentinel field, and logs `todo=0`. Right after a crash,
    wrong after a prompt edit.
    """
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    stale = [m for m in modules if PROMPT_FINGERPRINTS.get(m) is not None
             and stored.get(m) != PROMPT_FINGERPRINTS[m]]
    if not stale:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        for m in stale:
            print(f'  {m}: prompt changed since that checkpoint '
                  f'({stored.get(m) or "no fingerprint"} -> {PROMPT_FINGERPRINTS[m]})')
        print(f'  discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, modules) -> None:
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    for m in modules:
        if PROMPT_FINGERPRINTS.get(m) is not None:
            stored[m] = PROMPT_FINGERPRINTS[m]
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, modules, input_path=None, output_path=None,
              label='', extra_args=()) -> Path:
    """Run one pipeline stage as a subprocess. CLI flags override profile values."""
    spec = json.loads(profile.read_text(encoding='utf-8'))
    resolved_in = Path(input_path) if input_path else BASE / spec['input']
    resolved_out = Path(output_path) if output_path else BASE / spec['output']
    resolved_out.parent.mkdir(parents=True, exist_ok=True)
    assert resolved_in.stem.startswith('csd_'), (
        f'{resolved_in.name}: stage inputs must be named csd_* so their checkpoint stem '
        f'cannot collide with another notebook')
    assert resolved_in.exists(), f'stage input does not exist: {resolved_in}'

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(resolved_in), '--output', str(resolved_out), *map(str, extra_args)]
    banner = label or profile.stem
    print('=' * 78)
    print(f'STAGE: {banner}')
    print('=' * 78)
    _sync_checkpoint(resolved_in, modules)
    print(' '.join(str(c) for c in cmd), flush=True)

    err_lines = []

    def _drain(stream):
        for line in iter(stream.readline, ''):
            err_lines.append(line)

    t0 = time.time()
    proc = subprocess.Popen(cmd, text=True, encoding='utf-8', errors='replace',
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            bufsize=1, env=SUBPROCESS_ENV)
    t = threading.Thread(target=_drain, args=(proc.stderr,), daemon=True)
    t.start()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    t.join(timeout=5)
    if proc.returncode != 0:
        print(f'\nExit code {proc.returncode}')
        if err_lines:
            print('--- stderr (last 60 lines) ---')
            print(''.join(err_lines[-60:]))
        raise RuntimeError(f'{banner} failed with exit code {proc.returncode}')
    _stamp_checkpoint(resolved_in, modules)
    print(f'\n-> {banner} done in {time.time() - t0:.0f}s  ->  {resolved_out}')
    return resolved_out


print(f'\nCheckpoint guard armed. {MOD_EFFORT}: {PROMPT_FINGERPRINTS[MOD_EFFORT]}')

In [ ]:
# The full effort prompt, exactly as the model will receive it.
print(EFFORT_PROMPT)

## 6. Derived effort columns

`effort_score` is a five-level enum. Two derivations are added, and both are kept:

- `effort_score_num` (1–5) for correlation and ordinal work.
- `effort_high` (`high` or `very_high`) as the binary.

**The five levels are deliberately not collapsed.** Whether the middle of the scale carries
information is exactly the question the incoming survey data will answer; throwing away the
resolution now would make that impossible without a re-run.

In [ ]:
EFFORT_ORDER = ['very_low', 'low', 'medium', 'high', 'very_high']
EFFORT_NUM = {lab: i + 1 for i, lab in enumerate(EFFORT_ORDER)}
HIGH_LEVELS = {'high', 'very_high'}


def add_effort_derivations(df: pd.DataFrame) -> pd.DataFrame:
    """effort_score_num, effort_high, and the friction/theme indicator columns."""
    out = df.copy()
    s = out['effort_score'].astype(str).str.strip().str.lower()
    out['effort_score'] = s
    out['effort_score_num'] = s.map(EFFORT_NUM)
    out['effort_high'] = s.isin(HIGH_LEVELS)

    # friction_points arrives exploded into friction_points_1..N
    fcols = sorted([c for c in out.columns if re.match(r'^friction_points_\d+$', c)])
    if fcols:
        vals = out[fcols].astype(str)
        for cat in CANONICAL_FRICTION:
            out[f'fp_{re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")}'] = (
                (vals == cat).any(axis=1))
        out['friction_count'] = (vals.notna() & (vals != '') & (vals != 'nan')).sum(axis=1)
    return out


def effort_summary(df: pd.DataFrame, by: str, weights: str | None = None,
                   min_n: int = 30) -> pd.DataFrame:
    """Mean effort, %high and observed index by a grouping column.

    With `weights`, every figure is design-weighted to the population; without,
    it is the raw sample. Groups below min_n are kept but flagged, because a mean
    over eight tickets is not an estimate.
    """
    rows = []
    for val, g in df.groupby(by, dropna=False):
        w = g[weights] if weights else pd.Series(1.0, index=g.index)
        tot = w.sum()
        num = g['effort_score_num']
        ok = num.notna()
        rows.append({
            by: val, 'n': len(g),
            'est_population': tot if weights else np.nan,
            'mean_effort': float((num[ok] * w[ok]).sum() / w[ok].sum()) if ok.any() else np.nan,
            'pct_high': float((g['effort_high'] * w).sum() / tot),
            'obs_index': float((g['obs_effort_index'] * w).sum() / tot),
            'obs_span_h_median': float(g['obs_span_h'].median()),
            'estimable': len(g) >= min_n,
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False).reset_index(drop=True)


print('Derivations defined:', EFFORT_NUM)
print(f'effort_high = {sorted(HIGH_LEVELS)}')

## 7. Gate 1 — 40-ticket smoke test *(~$0.60)*

Cheap enough to repeat after any prompt edit. It checks that the enums come back valid, that
`friction_points` normalise onto the nine categories rather than free text, and that
`effort_evidence` is a real quote rather than a paraphrase — if the evidence is invented, the score
beside it cannot be trusted either.

In [ ]:
features_idx = features.set_index(ID_COL)
smoke_pool = _src.merge(features[[ID_COL, 'obs_effort_index']], on=ID_COL, how='left')
smoke_pool['_bucket'] = pd.qcut(smoke_pool['obs_effort_index'], 4,
                                labels=['q1', 'q2', 'q3', 'q4'], duplicates='drop')
smoke = stratified_sample(smoke_pool, '_bucket', SMOKE_N, min_per=8)
smoke[LLM_COLS].to_csv(EFFORT_SMOKE_CSV, index=False)
print(f'Smoke set: {len(smoke)} tickets, spread across observed-effort quartiles')
print(smoke['_bucket'].value_counts().sort_index().to_string())

smoke_out = run_stage(PROFILE_EFFORT, modules=[MOD_EFFORT], input_path=EFFORT_SMOKE_CSV,
                      output_path=SMOKE_DIR / 'csd_effort_smoke_output.xlsx',
                      label=f'GATE 1 - effort smoke, {len(smoke)} tickets')

In [ ]:
sm = add_effort_derivations(read_stage_output(smoke_out))
print(f'{len(sm)} rows x {len(sm.columns)} cols')
print('effort columns:', [c for c in sm.columns if c.startswith(('effort', 'friction', 'repeat',
                                                                 'resolution_transp'))])

_blank = sm['effort_score'].isna() | (sm['effort_score'].astype(str).str.strip() == '')
assert not _blank.any(), f'{int(_blank.sum())} blank effort_score on a {len(sm)}-ticket run'
_bad = ~sm['effort_score'].isin(EFFORT_ORDER)
assert not _bad.any(), f'effort_score outside the enum: {sm.loc[_bad, "effort_score"].tolist()}'
print('\neffort_score:'); print(sm['effort_score'].value_counts().reindex(EFFORT_ORDER).to_string())
print('\nresolution_transparency:'); print(sm['resolution_transparency'].value_counts().to_string())

# friction points must be canonical, not free text
# pandas 3 keeps NA through .astype(str) rather than rendering it 'nan', so filter
# on the actual values rather than their string form.
_fcols = sorted([c for c in sm.columns if re.match(r'^friction_points_\d+$', c)])
_flat = sm[_fcols].to_numpy().ravel() if _fcols else []
_filled = [v for v in _flat if isinstance(v, str) and v.strip() not in ('', 'nan', 'None')]
_noncanon = [v for v in _filled if v not in CANONICAL_FRICTION]
print(f'\nfriction_points columns: {_fcols}')
print(f'  values returned : {len(_filled)}  ({len(set(_filled))} distinct)')
print(f'  non-canonical   : {len(_noncanon)} ({len(_noncanon) / max(1, len(_filled)):.1%})')
if _noncanon:
    print(f'    {sorted(set(_noncanon))}')
    print('    A stray value means the model drew from the taxonomy labels instead of the')
    print('    friction list, or a wording variant is not covered by FRICTION_CATEGORIES.')
_rate = len(_noncanon) / max(1, len(_filled))
assert _rate <= 0.05, (
    f'{_rate:.0%} of friction values fall outside the nine categories: '
    f'{sorted(set(_noncanon))}. Either the model is ignoring the label list, or '
    f'friction_categories is missing from the config - re-run section 4.')

# evidence must be a real quote
from rca_pipeline.utils import _evidence_in_transcript
_texts = dict(zip(smoke[ID_COL].astype(str).str.strip(), smoke[TEXT_COL]))
_rows = sm[sm['effort_evidence'].astype(str).str.strip() != '']
_hits = sum(_evidence_in_transcript(str(r['effort_evidence']), _texts.get(r[ID_COL], ''))
            for _, r in _rows.iterrows())
_rate = _hits / max(1, len(_rows))
print(f'\neffort_evidence: {len(_rows)}/{len(sm)} non-empty, {_hits} found in the transcript '
      f'({_rate:.0%})')
assert _rate >= 0.80, (
    f'only {_rate:.0%} of evidence quotes are in the transcript - the model is paraphrasing, '
    f'so its scores cannot be trusted either')

In [ ]:
# Read these. Does the score match what the ticket actually shows?
_j = sm.merge(features[[ID_COL, 'obs_turns', 'obs_agent_handoffs', 'obs_span_h',
                        'obs_effort_index']], on=ID_COL, how='left')
for _, r in _j.sort_values('effort_score_num', ascending=False).head(5).iterrows():
    print('=' * 78)
    print(f'{r[ID_COL]}   effort={r["effort_score"]}  transparency={r["resolution_transparency"]}')
    print(f'  observed : {int(r["obs_turns"])} turns, {int(r["obs_agent_handoffs"])} handoff(s), '
          f'{r["obs_span_h"]:.0f}h span, index {r["obs_effort_index"]:.2f}')
    print(f'  why      : {r["effort_primary_description"]}')
    print(f'  friction : {[r[c] for c in _fcols if str(r.get(c, "")) not in ("", "nan")]}')
    print(f'  evidence : {str(r["effort_evidence"])[:200]}')

## 8. Gate 2 — 1,000-ticket probe *(~$7)*

The first measurement worth anything, at under a tenth of the main run's cost. The question it
answers: **does the model's judgement track the shape of the interaction at all?**

`effort_score` and `obs_effort_index` are independent readings — one from the transcript, one from
the thread record. They should be positively related. They should *not* be identical: the model can
see a customer who is angry after one exchange, and the record can see a ticket that took nine
turns for a purely administrative reason.

A Spearman correlation indistinguishable from zero would mean one of the two is not measuring
effort. That is worth finding out for $7.

**What this gate cannot tell you.** It measures the model against the *interaction record*, not
against a customer. Section 13 shows that the record itself tracks customer-reported effort only
weakly — `obs_effort_index` scores about +0.04 against the customers own answer. So a high
correlation here means the model is reading something real off the transcript; it does **not** mean
the model predicts how hard the customer found it. That question belongs to section 13.

In [ ]:
probe_pool = smoke_pool[~smoke_pool[ID_COL].isin(set(smoke[ID_COL]))]
probe = stratified_sample(probe_pool, '_bucket', PROBE_N, min_per=200, seed=RANDOM_SEED + 1)
probe[LLM_COLS].to_csv(EFFORT_PROBE_CSV, index=False)
print(f'Probe set: {len(probe)} tickets')
print(probe['_bucket'].value_counts().sort_index().to_string())

probe_out = run_stage(PROFILE_EFFORT, modules=[MOD_EFFORT], input_path=EFFORT_PROBE_CSV,
                      output_path=SMOKE_DIR / 'csd_effort_probe_output.xlsx',
                      label=f'GATE 2 - effort probe, {len(probe)} tickets')

In [ ]:
from scipy.stats import spearmanr

pr = add_effort_derivations(read_stage_output(probe_out)).merge(
    features, on=ID_COL, how='left')
print(f'{len(pr)} tickets, joined to observed features: {pr["obs_effort_index"].notna().mean():.1%}')

_blank = (pr['effort_score'].isna() | (pr['effort_score'].astype(str).str.strip() == '')).mean()
print(f'blank effort_score: {_blank:.2%}')
assert _blank <= 0.005, f'{_blank:.2%} blank effort scores'

print('\neffort_score distribution:')
_d = pr['effort_score'].value_counts().reindex(EFFORT_ORDER).fillna(0).astype(int)
for lab, n in _d.items():
    print(f'  {lab:<11} {n:>5}  {n / len(pr):>6.1%}  {"#" * int(60 * n / len(pr))}')
print(f'\neffort_high: {pr["effort_high"].mean():.1%}')

# ---- the gate: does judgement track the record? -------------------------
_ok = pr['effort_score_num'].notna() & pr['obs_effort_index'].notna()
RHO, PVAL = spearmanr(pr.loc[_ok, 'effort_score_num'], pr.loc[_ok, 'obs_effort_index'])
_n = int(_ok.sum())
_se = 1 / np.sqrt(_n - 3)
_lo, _hi = np.tanh(np.arctanh(RHO) - 1.96 * _se), np.tanh(np.arctanh(RHO) + 1.96 * _se)
print(f'\nSpearman(effort_score, obs_effort_index) = {RHO:.3f}  '
      f'95% CI [{_lo:.3f}, {_hi:.3f}]  n={_n:,}')

print('\nobserved index by LLM effort level:')
print(pr.groupby('effort_score')[['obs_effort_index', 'obs_turns', 'obs_agent_handoffs',
                                  'obs_span_h']]
      .agg(['mean']).reindex(EFFORT_ORDER)
      .to_string(float_format=lambda v: f'{v:,.2f}'))

assert _lo > 0, (
    f'the two readings are not positively related (95% CI includes zero). One of them is not '
    f'measuring effort. Read the examples in section 7 before spending the main run budget.')
if RHO < 0.15:
    print(f'\n  WEAK ({RHO:.2f}). Positive but slight. The two views are close to independent -'
          f'\n  report both separately rather than as one construct.')
else:
    print(f'\n  The two independent readings agree ({RHO:.2f}).')
print('\n  READ THIS BEFORE QUOTING THE NUMBER ABOVE. It measures agreement with')
print('  obs_effort_index - the shape of the interaction - NOT with the customer.')
print('  Section 13 scores that index against the customers own effort answer at')
print('  roughly +0.04. Agreeing with it is therefore agreeing with a weak proxy.')
print('  The number that decides whether effort scoring works is the LLM-vs-survey')
print('  correlation in section 13. This gate only confirms the model is reading')
print('  something real off the transcript rather than guessing.')

### Decision point

Before clearing the flag below, check:

- **The effort distribution is not degenerate.** If 90% of tickets land on one level, the score
  carries almost no information regardless of its correlation.
- **The Spearman CI excludes zero** — asserted above — and read its size. Anything under ~0.15 means
  the two views are nearly independent, which is a finding to report, not a number to average away.
- **The five worked examples in section 7 read correctly.** The observed columns are printed beside
  each score precisely so a wrong score is visible.

In [ ]:
STOP_AFTER_PROBE = False  # cleared 2026-08-07: Spearman vs the observed index 0.368
                          # [0.314, 0.421], monotonic across all five levels; distribution
                          # not degenerate (low 27% / medium 40% / high 32%); smoke evidence
                          # grounding 100%; friction values 100% canonical.

if STOP_AFTER_PROBE:
    raise SystemExit(
        f'Stopping after the probe. Spearman={RHO:.3f}, effort_high={pr["effort_high"].mean():.1%}. '
        f'Review, then set STOP_AFTER_PROBE = False and re-run from here. '
        f'The main run costs about ${len(_src) * 0.0068:,.0f}.')
print('Proceeding to the main run.')

## 9. Main run — the 12,000 triangulation cohort *(~$82)*

In [ ]:
effort_out = run_stage(PROFILE_EFFORT, modules=[MOD_EFFORT],
                       label=f'MAIN RUN - customer effort, {len(_src):,} tickets')

## 9b. Extend to the whole corpus *(~$260)*

The 2026-08-07 metadata refresh brought customer survey answers with it —
`CSAT_EFFORT_CODE`, on 3,554 tickets. Only 706 of those fall inside the 12,000 scored
above; scoring everything raises the validation set five-fold and tightens the
correlation in section 13 from roughly ±0.07 to ±0.03.

**The 12,000 are not paid for twice.** The checkpoint is a JSONL of per-uid results and
the pipeline treats any record whose sentinel field is already present as done. Copying
the completed checkpoint across, and stamping the same prompt fingerprint beside it,
means only the remaining ~38,700 tickets are sent. The copy is only sound because the
module, the prompt and the record text are identical — which is asserted before it
happens.

In [ ]:
RUN_FULL_CORPUS = True

FULL_INPUT = INTER_DIR / 'csd_effort_input_full.csv'
STAGE_08 = BASE / 'data' / 'csd_stage_input_2026.csv'

if not RUN_FULL_CORPUS:
    print('Skipped. The analysis runs on the 12,000 triangulation cohort.')
else:
    full = pd.read_csv(STAGE_08, dtype=str)
    assert list(full.columns) == LLM_COLS, f'unexpected columns in {STAGE_08.name}'
    # The 12,000 must be a strict subset, or seeding the checkpoint would be wrong.
    _missing = set(_src[ID_COL]) - set(full[ID_COL].astype(str).str.strip())
    assert not _missing, (
        f'{len(_missing)} of the scored 12,000 are absent from the full input - the '
        f'checkpoint cannot be reused')
    full.to_csv(FULL_INPUT, index=False)
    print(f'Full input: {len(full):,} tickets -> {FULL_INPUT.name}')

    # ---- seed the checkpoint so the 12,000 are not re-charged -------------
    src_ck = CHECKPOINT_DIR / f'{EFFORT_INPUT.stem}_checkpoint.jsonl'
    dst_ck = CHECKPOINT_DIR / f'{FULL_INPUT.stem}_checkpoint.jsonl'
    assert src_ck.exists(), f'{src_ck.name} not found - run section 9 first'
    _seed = [ln for ln in src_ck.read_text(encoding='utf-8').splitlines() if ln.strip()]
    if dst_ck.exists() and dst_ck.stat().st_size >= src_ck.stat().st_size:
        print(f'{dst_ck.name} already at least as complete as the seed - left alone')
    else:
        dst_ck.write_text('\n'.join(_seed) + '\n', encoding='utf-8')
        print(f'Seeded {dst_ck.name} with {len(_seed):,} completed records')
    # Stamp the fingerprint too, or the guard discards what we just copied.
    _stamp_checkpoint(FULL_INPUT, [MOD_EFFORT])

    _todo = len(full) - len(_seed)
    print(f'\nRecords still to score: ~{_todo:,}  (~${_todo * 0.0068:,.0f})')

    effort_out = run_stage(
        PROFILE_EFFORT, modules=[MOD_EFFORT], input_path=FULL_INPUT,
        output_path=OUT_DIR / 'csd_2026_customer_effort_full.xlsx',
        label=f'FULL CORPUS - customer effort, {len(full):,} tickets')

## 10. Merge — effort, observed features, rules, reasons, codes

Three sources joined on `INCIDENT_REF`, all covering exactly the same 12,000 tickets:

| source | brings |
|---|---|
| this run | `effort_score`, friction points, effort themes, transparency, evidence |
| `csd_effort_features.csv` | the `obs_*` measurements |
| notebook 08's output | validation rule, rejection reason, resolution code, conformance |

The design weights are recomputed here exactly as in `scripts/build_deck_data.py` — inverse
probability `N_h / n_h` by resolution code — so effort extrapolates to the 50,721 classifiable
tickets on the same basis as everything else.

In [ ]:
SCORED = add_effort_derivations(read_stage_output(effort_out))
SCORED = SCORED.merge(features, on=ID_COL, how='left', validate='one_to_one')
assert SCORED['obs_effort_index'].notna().all(), 'a scored ticket has no observed features'
print(f'Scored for effort: {len(SCORED):,} tickets')

# Rules and reasons exist only for the 12,000 notebook 08 tagged. The triangulation
# frame is the intersection; SCORED stays available for effort-only and survey work
# across whatever was scored.
_keep = [c for c in tagged.columns if c not in SCORED.columns or c == ID_COL]
EFFORT = SCORED.merge(tagged[_keep], on=ID_COL, how='inner', validate='one_to_one')

# A handful of tickets came back from notebook 08 with a validation-rule code that is
# not in the taxonomy (CSDVR6 and friends), so derive_labels left the label blank.
# Bucket them explicitly: dropping them silently would unbalance the design weights,
# and the rate is worth carrying into the output.
INVALID_LABEL = '(invalid code returned by the model)'
for _c in ('rule_label', 'reason_label'):
    _n = int(EFFORT[_c].isna().sum())
    if _n:
        print(f'  {_n} ticket(s) have no {_c} - bucketed as {INVALID_LABEL!r} '
              f'({_n / len(EFFORT):.2%})')
        EFFORT[_c] = EFFORT[_c].fillna(INVALID_LABEL)
assert EFFORT[RC_LABEL].notna().all(), (
    'a triangulation ticket has no resolution code - it cannot be weighted')
print(f'Triangulation cohort (effort + rule + reason): {len(EFFORT):,} tickets')
if len(SCORED) > len(EFFORT):
    print(f'  {len(SCORED) - len(EFFORT):,} further tickets have effort but no rule/reason - '
          f'used in section 13 only')

# ---- design weights, identical basis to the deck ------------------------
_exc = pd.read_csv(INTER_DIR / 'csd_2026_excluded.csv', dtype=str, usecols=[ID_COL])
_all = pd.read_csv(DATA_DIR / 'scopus content_2026_threads_collapsed.csv', dtype=str,
                   usecols=[ID_COL])
_meta = pd.read_csv(META_TSV, sep='\t', dtype=str,
                    usecols=['incident_reference', 'resolution_code_lvl1_desc'])
_pop = (_all[~_all[ID_COL].isin(set(_exc[ID_COL]))]
        .merge(_meta, left_on=ID_COL, right_on='incident_reference', how='left'))
# A ticket with no resolution code belongs to no stratum and cannot be weighted.
# The 2026-08-07 metadata refresh dropped a handful of references that are still in
# the conversations file; they are removed here rather than silently unbalancing
# the weights.
_no_code = _pop['resolution_code_lvl1_desc'].isna()
if _no_code.any():
    print(f'{int(_no_code.sum())} classifiable ticket(s) have no resolution code in the '
          f'current metadata - excluded from the weighting population')
_pop = _pop[~_no_code]
N_POP = len(_pop)
N_h = _pop['resolution_code_lvl1_desc'].value_counts()
n_h = EFFORT[RC_LABEL].value_counts()
EFFORT['w'] = EFFORT[RC_LABEL].map(N_h) / EFFORT[RC_LABEL].map(n_h)
assert abs(EFFORT['w'].sum() - N_POP) < 1e-6, (
    f'weights sum to {EFFORT["w"].sum():,.2f}, expected {N_POP:,}')
print(f'Design weights sum to {EFFORT["w"].sum():,.0f} = the classifiable population  OK')

print(f'\neffort_score across the sample:')
_d = EFFORT['effort_score'].value_counts().reindex(EFFORT_ORDER).fillna(0).astype(int)
_w = EFFORT.groupby('effort_score')['w'].sum().reindex(EFFORT_ORDER).fillna(0)
for lab in EFFORT_ORDER:
    print(f'  {lab:<11} sample {_d[lab]:>6,} ({_d[lab] / len(EFFORT):>5.1%})   '
          f'population {_w[lab]:>8,.0f} ({_w[lab] / N_POP:>5.1%})')
print(f'\neffort_high: sample {EFFORT["effort_high"].mean():.1%}, '
      f'population {(EFFORT["effort_high"] * EFFORT["w"]).sum() / N_POP:.1%}')

## 11. Triangulation

The point of the exercise: effort set against the rule that made the request invalid, the reason
the agent gave, and the code they logged. Every population figure is design-weighted.

### 11.1 Effort by rejection reason

In [ ]:
BY_REASON = effort_summary(EFFORT, 'reason_label', weights='w')
print(BY_REASON.to_string(index=False, formatters={
    'est_population': '{:,.0f}'.format, 'mean_effort': '{:.2f}'.format,
    'pct_high': '{:.1%}'.format, 'obs_index': '{:.3f}'.format,
    'obs_span_h_median': '{:,.0f}'.format}))
_est = BY_REASON[BY_REASON.estimable]
if len(_est) >= 2:
    _hi, _lo = _est.iloc[_est.pct_high.argmax()], _est.iloc[_est.pct_high.argmin()]
    print(f'\nWidest gap: "{_hi.reason_label}" at {_hi.pct_high:.0%} high-effort '
          f'({_hi.est_population:,.0f} tickets/yr) versus "{_lo.reason_label}" at '
          f'{_lo.pct_high:.0%}.')

### 11.2 Effort by validation rule, and by resolution code

In [ ]:
BY_RULE = effort_summary(EFFORT, 'rule_label', weights='w')
_r = BY_RULE.copy(); _r['rule_label'] = _r['rule_label'].str.slice(0, 46)
print('BY VALIDATION RULE (top 12 by volume):')
print(_r.head(12).to_string(index=False, formatters={
    'est_population': '{:,.0f}'.format, 'mean_effort': '{:.2f}'.format,
    'pct_high': '{:.1%}'.format, 'obs_index': '{:.3f}'.format,
    'obs_span_h_median': '{:,.0f}'.format}))

BY_CODE = effort_summary(EFFORT, RC_LABEL, weights='w')
print('\n\nBY RESOLUTION CODE:')
print(BY_CODE.to_string(index=False, formatters={
    'est_population': '{:,.0f}'.format, 'mean_effort': '{:.2f}'.format,
    'pct_high': '{:.1%}'.format, 'obs_index': '{:.3f}'.format,
    'obs_span_h_median': '{:,.0f}'.format}))

### 11.3 Does the matrix's failure cost customers effort?

Two questions the earlier analysis could not answer on its own:

1. Tickets where the model's rule and the agent's code **disagree** — is that disagreement
   associated with a harder experience for the customer, or is it purely an internal bookkeeping
   matter?
2. The **no-applicable-rule** segment — the tickets the matrix does not cover. If those are also
   the high-effort ones, the gaps found earlier are not just a governance tidy-up.

In [ ]:
rows = []
for name, mask in [
    ('Rule matches the logged code', EFFORT['conf_strict'] & ~EFFORT['rule_is_na']),
    ('Rule and code disagree', ~EFFORT['conf_strict'] & ~EFFORT['rule_is_na']
     & EFFORT['is_mappable']),
    ('No applicable rule (CSDVR4.1)', EFFORT['rule_is_na']),
    ('Code no rule can claim', ~EFFORT['is_mappable']),
]:
    g = EFFORT[mask]
    if not len(g):
        continue
    w = g['w']
    rows.append({'segment': name, 'n': len(g), 'est_population': w.sum(),
                 'pct_high': (g['effort_high'] * w).sum() / w.sum(),
                 'mean_effort': (g['effort_score_num'] * w).sum() / w.sum(),
                 'obs_index': (g['obs_effort_index'] * w).sum() / w.sum(),
                 'span_h_median': g['obs_span_h'].median(),
                 'repeat_rate': g['obs_repeat_contact'].mean()})
CONFORMANCE_EFFORT = pd.DataFrame(rows)
print(CONFORMANCE_EFFORT.to_string(index=False, formatters={
    'est_population': '{:,.0f}'.format, 'pct_high': '{:.1%}'.format,
    'mean_effort': '{:.2f}'.format, 'obs_index': '{:.3f}'.format,
    'span_h_median': '{:,.0f}'.format, 'repeat_rate': '{:.1%}'.format}))

_base = CONFORMANCE_EFFORT.loc[CONFORMANCE_EFFORT.segment ==
                               'Rule matches the logged code', 'pct_high']
if len(_base):
    b = float(_base.iloc[0])
    print(f'\nAgainst a {b:.1%} high-effort baseline where rule and code agree:')
    for _, r in CONFORMANCE_EFFORT.iterrows():
        if r.segment == 'Rule matches the logged code':
            continue
        print(f'  {r.segment:<32} {r.pct_high:>6.1%}  ({(r.pct_high - b) * 100:+.1f}pp, '
              f'{r.est_population:,.0f} tickets/yr)')

### 11.4 Friction points and effort themes

In [ ]:
fp_cols = [c for c in EFFORT.columns if c.startswith('fp_')]
rows = []
for c in fp_cols:
    lab = next(x for x in CANONICAL_FRICTION
               if f'fp_{re.sub(r"[^a-z0-9]+", "_", x.lower()).strip("_")}' == c)
    w = EFFORT['w']
    rows.append({'friction_point': lab, 'n': int(EFFORT[c].sum()),
                 'sample_pct': EFFORT[c].mean(),
                 'est_population': float((EFFORT[c] * w).sum()),
                 'pop_pct': float((EFFORT[c] * w).sum() / N_POP),
                 'pct_high_when_present': float(
                     (EFFORT.loc[EFFORT[c], 'effort_high'] * w[EFFORT[c]]).sum()
                     / max(1e-9, w[EFFORT[c]].sum()))})
FRICTION = pd.DataFrame(rows).sort_values('est_population', ascending=False)
print('FRICTION POINTS (the ninth category is the one added for this corpus):')
print(FRICTION.to_string(index=False, formatters={
    'sample_pct': '{:.1%}'.format, 'est_population': '{:,.0f}'.format,
    'pop_pct': '{:.1%}'.format, 'pct_high_when_present': '{:.1%}'.format}))

print('\n\nFRICTION POINT x REJECTION REASON (population estimates, top reasons):')
_top = BY_REASON.head(6)['reason_label'].tolist()
_x = pd.DataFrame({
    lab: [float((EFFORT.loc[EFFORT.reason_label == r, c] *
                 EFFORT.loc[EFFORT.reason_label == r, 'w']).sum())
          for r in _top]
    for c, lab in zip(fp_cols, [next(x for x in CANONICAL_FRICTION
                                     if f'fp_{re.sub(chr(91) + "^a-z0-9" + chr(93) + "+", "_", x.lower()).strip("_")}' == c)
                                for c in fp_cols])}, index=_top)
print(_x.to_string(float_format=lambda v: f'{v:,.0f}'))

### 11.5 Effort themes

In [ ]:
tcols = sorted([c for c in EFFORT.columns if re.match(r'^effort_themes_\d+$', c)])
_long = EFFORT[[ID_COL, 'w'] + tcols].melt(id_vars=[ID_COL, 'w'], value_name='theme')
_long = _long[_long.theme.notna() & (_long.theme.astype(str).str.strip() != '')]
_long['label'] = _long.theme.map(EFFORT_LEAVES).fillna(_long.theme)
THEMES = (_long.groupby('label').agg(n=('w', 'size'), est_population=('w', 'sum'))
          .sort_values('est_population', ascending=False).reset_index())
THEMES['pop_pct'] = THEMES.est_population / N_POP
print(f'{len(tcols)} theme slots; {len(THEMES)} distinct themes used')
print(THEMES.to_string(index=False, formatters={
    'est_population': '{:,.0f}'.format, 'pop_pct': '{:.1%}'.format}))

## 12. Judged effort against observed effort

The two independent readings, on the full run. **Neither is ground truth**, and the observed index
turns out to be a poor stand-in for one — section 13 puts it at about +0.04 against the customer's
own answer.

So read this section as a coherence check: does the model's reading of the transcript move with the
measurable shape of the interaction? Where it does, the model is not hallucinating. Where it does
not is the more interesting half, and section 13 is what adjudicates.

In [ ]:
from scipy.stats import spearmanr

_ok = EFFORT['effort_score_num'].notna() & EFFORT['obs_effort_index'].notna()
RHO_FULL, _ = spearmanr(EFFORT.loc[_ok, 'effort_score_num'],
                        EFFORT.loc[_ok, 'obs_effort_index'])
_n = int(_ok.sum()); _se = 1 / np.sqrt(_n - 3)
CI_FULL = (np.tanh(np.arctanh(RHO_FULL) - 1.96 * _se),
           np.tanh(np.arctanh(RHO_FULL) + 1.96 * _se))
print(f'Spearman(judged, observed) = {RHO_FULL:.3f}  '
      f'95% CI [{CI_FULL[0]:.3f}, {CI_FULL[1]:.3f}]  n={_n:,}')

EFFORT['obs_decile'] = pd.qcut(EFFORT['obs_effort_index'], 10, labels=False,
                               duplicates='drop') + 1
DECILES = (EFFORT.groupby('obs_decile')
           .apply(lambda g: pd.Series({
               'n': len(g),
               'obs_index': g['obs_effort_index'].mean(),
               'pct_high_judged': (g['effort_high'] * g['w']).sum() / g['w'].sum(),
               'mean_judged': (g['effort_score_num'] * g['w']).sum() / g['w'].sum(),
               'span_h_median': g['obs_span_h'].median()}), include_groups=False)
           .reset_index())
print('\nJudged effort by observed-effort decile:')
print(DECILES.to_string(index=False, formatters={
    'n': '{:,.0f}'.format, 'obs_index': '{:.3f}'.format,
    'pct_high_judged': '{:.1%}'.format, 'mean_judged': '{:.2f}'.format,
    'span_h_median': '{:,.0f}'.format}))

_d1, _d10 = DECILES.iloc[0], DECILES.iloc[-1]
print(f'\nBottom observed decile: {_d1.pct_high_judged:.1%} judged high-effort')
print(f'Top observed decile   : {_d10.pct_high_judged:.1%}')
print(f'Spread                : {_d10.pct_high_judged - _d1.pct_high_judged:+.1%}')
print('\nNeither of these two readings is a customer. Section 13 is where a customer speaks -')
print('and it shows the observed index tracking their answer at roughly +0.04, so treat the')
print('agreement above as evidence the model reads the transcript coherently, not as evidence')
print('it predicts perceived effort.')

In [ ]:
# Where they disagree most - worth reading, and the input to any reweighting later.
EFFORT['judged_pct'] = EFFORT['effort_score_num'].rank(pct=True)
EFFORT['divergence'] = EFFORT['judged_pct'] - EFFORT['obs_effort_index']
_hi_judged = EFFORT.nlargest(3, 'divergence')
_hi_obs = EFFORT.nsmallest(3, 'divergence')
for title, rows_ in [('Model says HIGH effort, the record looks light', _hi_judged),
                     ('The record looks heavy, the model says LOW effort', _hi_obs)]:
    print('=' * 78); print(title); print('=' * 78)
    for _, r in rows_.iterrows():
        print(f'{r[ID_COL]}  judged={r["effort_score"]}  obs_index={r["obs_effort_index"]:.2f}  '
              f'({int(r["obs_turns"])} turns, {r["obs_span_h"]:.0f}h)')
        print(f'   why: {r["effort_primary_description"]}')
        print(f'   ev : {str(r["effort_evidence"])[:170]}')
    print()

## 13. Validate against the customer survey

The 2026-08-07 metadata refresh brought `CSAT_EFFORT_CODE` with it — the customer's own
answer to a Customer Effort Score statement. **This is the only measure here that comes
from a customer.** Everything else is either the model reading the transcript or the
thread record counting turns.

The scale is agreement with "it was easy": `1` Strongly Agree … `5` Strongly Disagree, so
a **higher code means the customer found it harder**. Codes `0` (No Value) and `6`
(Do Not Know) are non-responses and are excluded rather than treated as midpoints.

Two things get tested against it: the model's judgement, and the observed index. They are
reported side by side because the comparison is the finding.

In [ ]:
from scipy.stats import spearmanr

SURVEY_COLS = ['CSAT_EFFORT_CODE', 'CSAT_EFFORT_DESC', 'CSAT_ANSWER_CODE', 'CSAT_DSAT_REASON']
_meta_full = pd.read_csv(META_TSV, sep='\t', dtype=str)
_have = [c for c in SURVEY_COLS if c in _meta_full.columns]
if not _have:
    print('No CSAT columns in the metadata - nothing to validate against.')
    SURVEY = None
else:
    _meta_full['ref'] = _meta_full['incident_reference'].astype(str).str.strip()
    SURVEY = SCORED.merge(_meta_full[['ref'] + _have], left_on=ID_COL, right_on='ref',
                          how='left')
    SURVEY['survey_code'] = pd.to_numeric(SURVEY['CSAT_EFFORT_CODE'], errors='coerce')
    # 0 = No Value, 6 = Do Not Know. Neither is a point on the scale.
    SURVEY['survey_valid'] = SURVEY['survey_code'].between(1, 5)
    SURVEY['survey_hard'] = SURVEY['survey_code'] >= 4      # Disagree / Strongly Disagree
    V = SURVEY[SURVEY['survey_valid'] & SURVEY['effort_score_num'].notna()]
    print(f'scored tickets                    : {len(SCORED):,}')
    print(f'  with any CSAT response          : {int(SURVEY["survey_code"].notna().sum()):,}')
    print(f'  with a usable effort answer 1-5 : {len(V):,}  ({len(V) / len(SCORED):.1%})')
    print(f'  of whom said it was hard (4-5)  : {int(V["survey_hard"].sum()):,} '
          f'({V["survey_hard"].mean():.1%})')

In [ ]:
def _rho(a, b):
    ok = pd.Series(a).notna() & pd.Series(b).notna()
    n = int(ok.sum())
    if n < 30:
        return float('nan'), float('nan'), float('nan'), n
    r, _ = spearmanr(pd.Series(a)[ok], pd.Series(b)[ok])
    se = 1 / np.sqrt(n - 3)
    return r, np.tanh(np.arctanh(r) - 1.96 * se), np.tanh(np.arctanh(r) + 1.96 * se), n


if SURVEY is not None and len(V) >= 30:
    CANDIDATES = [('LLM judged effort_score', 'effort_score_num'),
                  ('obs_effort_index (composite)', 'obs_effort_index'),
                  ('obs_customer_turns', 'obs_customer_turns'),
                  ('obs_turns', 'obs_turns'),
                  ('obs_span_h', 'obs_span_h'),
                  ('obs_agent_handoffs', 'obs_agent_handoffs'),
                  ('obs_first_response_h', 'obs_first_response_h'),
                  ('obs_repeat_contact', 'obs_repeat_contact')]
    rows = []
    for name, col in CANDIDATES:
        vals = V[col].astype('float') if V[col].dtype.name == 'boolean' else V[col]
        r, lo, hi, n = _rho(vals, V['survey_code'])
        rows.append({'signal': name, 'spearman': r, 'ci_lo': lo, 'ci_hi': hi, 'n': n})
    SURVEY_VALIDATION = pd.DataFrame(rows).sort_values('spearman', ascending=False)
    print('Agreement with the customer\'s own effort answer '
          '(higher survey code = customer found it harder):\n')
    print(SURVEY_VALIDATION.to_string(index=False, formatters={
        'spearman': '{:+.3f}'.format, 'ci_lo': '{:+.3f}'.format,
        'ci_hi': '{:+.3f}'.format, 'n': '{:,}'.format}))

    _llm = SURVEY_VALIDATION.iloc[0]
    _idx = SURVEY_VALIDATION[SURVEY_VALIDATION.signal.str.startswith('obs_effort_index')].iloc[0]
    print(f'\nBest signal: {_llm.signal} at {_llm.spearman:+.3f}')
    print(f'Composite index: {_idx.spearman:+.3f}')
    if _llm.signal.startswith('LLM') and _llm.spearman > _idx.spearman + 0.05:
        print('\n  What the customer experienced as effort is visible in what they wrote,')
        print('  not in how the ticket was handled. Turn counts, handoffs and elapsed time')
        print('  are close to uninformative about perceived effort on this corpus.')

    print('\n\nSurvey answer by judged effort level:')
    print(V.groupby('effort_score').agg(
        n=('survey_code', 'size'), mean_survey=('survey_code', 'mean'),
        pct_hard=('survey_hard', 'mean')).reindex(EFFORT_ORDER).dropna(how='all')
        .to_string(float_format=lambda x: f'{x:.2f}'))

    print('\n\nJudged high-effort vs customer said hard:')
    _ct = pd.crosstab(V['effort_high'], V['survey_hard'], margins=True)
    print(_ct.to_string())
    _tp = int(_ct.loc[True, True]) if True in _ct.index else 0
    _fn = int(_ct.loc[False, True]) if False in _ct.index else 0
    _fp = int(_ct.loc[True, False]) if True in _ct.index else 0
    if _tp + _fn:
        print(f'\n  recall    {_tp / (_tp + _fn):.0%}  (of customers who said it was hard, '
              f'how many the model flagged)')
    if _tp + _fp:
        print(f'  precision {_tp / (_tp + _fp):.0%}  (of those flagged, how many said so)')
    print('\n  Precision is bounded by how rare "hard" answers are - only '
          f'{V["survey_hard"].mean():.0%} of respondents gave one. Read recall first.')
else:
    SURVEY_VALIDATION = None
    print('Too few survey responses among the scored tickets to say anything.')

In [ ]:
# Does the model track the customer within each rejection reason, or only overall?
# Only answerable on the triangulation cohort, and only where enough respondents exist.
if SURVEY is not None and SURVEY_VALIDATION is not None:
    VT = V[V[ID_COL].isin(set(EFFORT[ID_COL]))].merge(
        tagged[[ID_COL, 'reason_label', 'rule_label']], on=ID_COL, how='left')
    rows = []
    for reason, g in VT.groupby('reason_label'):
        if len(g) < 40:
            continue
        r, lo, hi, n = _rho(g['effort_score_num'], g['survey_code'])
        rows.append({'rejection_reason': str(reason)[:44], 'n': n, 'spearman': r,
                     'ci_lo': lo, 'ci_hi': hi,
                     'pct_hard': g['survey_hard'].mean(),
                     'pct_judged_high': g['effort_high'].mean()})
    if rows:
        BY_REASON_SURVEY = pd.DataFrame(rows).sort_values('n', ascending=False)
        print('Within each rejection reason (>=40 respondents):\n')
        print(BY_REASON_SURVEY.to_string(index=False, formatters={
            'spearman': '{:+.3f}'.format, 'ci_lo': '{:+.3f}'.format,
            'ci_hi': '{:+.3f}'.format, 'pct_hard': '{:.1%}'.format,
            'pct_judged_high': '{:.1%}'.format}))
    else:
        BY_REASON_SURVEY = None
        print('No rejection reason has 40+ survey respondents in the triangulation cohort.')
else:
    BY_REASON_SURVEY = None

## 13b. How repeatable is the score?

The 12,000 triangulation cohort was scored **twice** — once on its own, and again as part
of the full-corpus run, which did not reuse the earlier results. That was not planned, and
it is the only repeat measurement in this work. It is also the most informative thing here
about how much any single effort number should be trusted.

Two runs of the same prompt, on the same model, over the same tickets, do not agree with
each other. What follows is the ceiling on how well this instrument can correlate with
*anything* — including the customer.

Note what this does **not** mean: the aggregate picture is stable enough to act on, and
every relationship in sections 11–13 survives. What it means is that a single ticket's
effort score is not a fact about that ticket, and the confidence intervals quoted
elsewhere describe sampling error only — they say nothing about the model's own
variability.

In [ ]:
RETEST_A = OUT_DIR / 'csd_2026_customer_effort.xlsx'          # 12,000-only run
RETEST_B = OUT_DIR / 'csd_2026_customer_effort_full.xlsx'     # full-corpus run

RELIABILITY = None
if RETEST_A.exists() and RETEST_B.exists():
    _a = pd.read_excel(RETEST_A, usecols=[ID_COL, 'effort_score', 'resolution_transparency'])
    _b = pd.read_excel(RETEST_B, usecols=[ID_COL, 'effort_score', 'resolution_transparency'])
    for _d in (_a, _b):
        _d[ID_COL] = _d[ID_COL].astype(str).str.strip()
    _m = _a.merge(_b, on=ID_COL, suffixes=('_1', '_2'))
    for _c in ('effort_score_1', 'effort_score_2'):
        _m[_c + '_n'] = _m[_c].astype(str).str.strip().str.lower().map(EFFORT_NUM)
    _m = _m.dropna(subset=['effort_score_1_n', 'effort_score_2_n'])

    _exact = float((_m.effort_score_1 == _m.effort_score_2).mean())
    _within1 = float(((_m.effort_score_1_n - _m.effort_score_2_n).abs() <= 1).mean())
    _rho, _ = spearmanr(_m.effort_score_1_n, _m.effort_score_2_n)
    _h1 = _m.effort_score_1.isin(HIGH_LEVELS)
    _h2 = _m.effort_score_2.isin(HIGH_LEVELS)
    _obs = float((_h1 == _h2).mean())
    _exp = _h1.mean() * _h2.mean() + (1 - _h1.mean()) * (1 - _h2.mean())
    _kappa = (_obs - _exp) / (1 - _exp)
    _trans = float((_m.resolution_transparency_1 == _m.resolution_transparency_2).mean())

    RELIABILITY = pd.DataFrame([
        {'measure': 'tickets scored twice', 'value': f'{len(_m):,}'},
        {'measure': 'exact agreement, 5-level score', 'value': f'{_exact:.1%}'},
        {'measure': 'agreement within one level', 'value': f'{_within1:.1%}'},
        {'measure': 'Spearman run 1 vs run 2', 'value': f'{_rho:+.3f}'},
        {'measure': 'high-effort rate, run 1', 'value': f'{_h1.mean():.1%}'},
        {'measure': 'high-effort rate, run 2', 'value': f'{_h2.mean():.1%}'},
        {'measure': 'binary agreement (high vs not)', 'value': f'{_obs:.1%}'},
        {'measure': "Cohen's kappa, binary", 'value': f'{_kappa:.3f}'},
        {'measure': 'resolution_transparency exact agreement', 'value': f'{_trans:.1%}'},
    ])
    print(RELIABILITY.to_string(index=False))

    print(f'\nWhat this bounds:')
    print(f'  Two runs of the same prompt correlate at {_rho:+.3f}. That is the ceiling on how')
    print(f'  well effort_score can correlate with anything measured on these tickets.')
    if SURVEY_VALIDATION is not None:
        _v = float(SURVEY_VALIDATION.iloc[0]['spearman'])
        print(f'  Measured against the customer: {_v:+.3f}.')
        print(f'  Corrected for that unreliability: about {_v / (_rho ** 0.5):+.3f}.')
    print(f'\n  The high-effort rate moved {abs(_h1.mean() - _h2.mean()) * 100:.1f}pp between')
    print(f'  runs. Every confidence interval in this notebook is a SAMPLING interval and does')
    print(f'  not include that. Quote aggregate effort rates to the nearest few points, and do')
    print(f'  not use a single ticket\'s effort score as a fact about that ticket.')
else:
    print('Only one scoring run on disk - no repeat measurement available.')

## 14. Write the deliverable

Sheet `survey_validation` is the one to read first if you only read one: it is the only
place in this analysis where a customer's own answer appears.

In [ ]:
XL_LIMIT = 32_767


def _clip(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].map(lambda v: v[:XL_LIMIT - 20] + ' ...[TRUNCATED]'
                                if isinstance(v, str) and len(v) > XL_LIMIT else v)
    return out


TICKET_COLS = [c for c in [
    ID_COL, RC_LABEL, 'rule_label', 'reason_label',
    'effort_score', 'effort_score_num', 'effort_high', 'effort_primary_code',
    'effort_primary_label', 'effort_primary_description', 'effort_evidence',
    'resolution_transparency', 'repeat_contact', 'friction_count',
    'obs_turns', 'obs_customer_turns', 'obs_agent_handoffs', 'obs_first_response_h',
    'obs_span_h', 'obs_repeat_contact', 'obs_prior_tickets_90d', 'obs_effort_index',
    'conf_strict', 'conf_loose', 'is_mappable', 'rule_is_na', 'w',
] if c in EFFORT.columns] + [c for c in EFFORT.columns
                             if re.match(r'^(friction_points|effort_themes)_\d+$', c)]

SHEETS = {
    'tickets': EFFORT[TICKET_COLS],
    'by_rejection_reason': BY_REASON,
    'by_validation_rule': BY_RULE,
    'by_resolution_code': BY_CODE,
    'conformance_x_effort': CONFORMANCE_EFFORT,
    'friction_points': FRICTION,
    'effort_themes': THEMES,
    'judged_vs_observed': DECILES,
    **({'reliability': RELIABILITY}
       if 'RELIABILITY' in dir() and RELIABILITY is not None else {}),
    **({'survey_validation': SURVEY_VALIDATION}
       if 'SURVEY_VALIDATION' in dir() and SURVEY_VALIDATION is not None else {}),
    **({'survey_by_reason': BY_REASON_SURVEY}
       if 'BY_REASON_SURVEY' in dir() and BY_REASON_SURVEY is not None else {}),
    'method': pd.DataFrame([
        {'item': 'Tickets scored', 'value': f'{len(EFFORT):,}'},
        {'item': 'Extrapolated to', 'value': f'{N_POP:,} classifiable 2026 tickets'},
        {'item': 'Weighting', 'value': 'inverse probability N_h/n_h by resolution code'},
        {'item': 'Spearman judged vs observed',
         'value': f'{RHO_FULL:.3f} [{CI_FULL[0]:.3f}, {CI_FULL[1]:.3f}]'},
        {'item': 'channel_switches', 'value': 'dropped - channel never varies within a ticket'},
        {'item': 'Friction categories', 'value': f'{len(CANONICAL_FRICTION)} '
         f'(added "{NEW_CATEGORY}")'},
        {'item': 'Ground truth',
         'value': ('CSAT_EFFORT_CODE from the 2026-08-07 metadata refresh - the customer\'s '
                   'own answer. See the survey_validation sheet.')
         if 'SURVEY_VALIDATION' in dir() and SURVEY_VALIDATION is not None
         else 'NONE for these tickets - see section 13.'},
        {'item': 'obs_effort_index', 'value': 'authored equal-weight composite of four '
         'percentile-ranked components, NOT a fitted model and NOT a good proxy for '
         'customer-reported effort - see the survey_validation sheet'},
    ]),
}

with pd.ExcelWriter(TRIANGULATION_XLSX, engine='openpyxl') as w:
    for name, df in SHEETS.items():
        _clip(df).to_excel(w, sheet_name=name[:31], index=False)
print(f'Wrote {TRIANGULATION_XLSX.relative_to(BASE)}')
for name, df in SHEETS.items():
    print(f'  {name:<22} {len(df):>7,} rows x {len(df.columns):>3} cols')

## Afterwards

- **Notebook 08 still runs.** The config now holds four modules; nothing of its was removed. Its
  checkpoints are untouched because this notebook used its own input file.
- **`friction_categories` is wiped by any rebuild.** Section 4's last cell must be re-run after
  `rca build build`, or friction normalisation silently reverts to a no-op — which is its
  state in every other config in this repo.
- **To restore the pre-effort state**, see `generated/_backup_pre_csd_effort/RESTORE.txt`.
- **To extend to all 50,721 tickets**, point the profile at `data/csd_stage_input_2026.csv` and
  re-run section 9. About $345, and worth it if the survey covers the full corpus.